# 04 — Forecasting con ARIMA y SARIMA
Notebook de aprendizaje: vamos a predecir las ventas diarias de **una sola serie** con modelos estadísticos clásicos (ARIMA y SARIMA), paso a paso.

**¿Qué es una serie de tiempo?** Una lista de mediciones de la misma cosa, tomadas en orden y a intervalos regulares. Acá: las unidades vendidas de un producto en una tienda, **un dato por día**.

La diferencia con un dataset "normal" (tabular) es clave:
- En un dataset tabular, cada fila es independiente: el orden de las filas no importa y se pueden mezclar.
- En una serie de tiempo, **el orden ES la información**. Lo que se vendió ayer ayuda a predecir lo que se vende hoy. Por eso nunca se mezclan las filas ni se parte al azar (ver DEC-002).

**Plan del notebook** (un paso por sección):
1. Elegir una serie y graficarla. ✔
2. Descomponerla en tendencia, estacionalidad y ruido. ✔
3. Estacionariedad.
   - A. Diagnóstico: gráficos y tests ADF/KPSS sobre la serie original. ✔
   - B. Diferenciar (simple y estacional) y elegir `d` y `D`.
4. Leer los gráficos ACF/PACF para elegir los órdenes del modelo.
5. Ajustar un SARIMA y medir el error sobre `valid`.
6. Compararlo con el modelo de boosting de la solución del curso.

Los fundamentos de cada paso están explicados en [`01_Documentos/fundamentos_series_tiempo.md`](../01_Documentos/fundamentos_series_tiempo.md).

**Reglas del notebook**
- Se explora y se entrena **solo con `train`**. `valid` se usa recién en el paso 5, para medir. `reserva` no se toca.
- Las observaciones se escriben **después de ver los números**.
- Las decisiones con alternativa descartada están en `decisions.md` (resumen) y `decisions_detalle.md` (detalle).

## 0. Preparación
Mismas convenciones que el notebook 03: rutas con `Path` y nombres de columnas en constantes, así las funciones no tienen valores fijos adentro (DEC-005).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)

DATOS = (Path("..") / "02_datos").resolve()
RUTA_TRAIN = DATOS / "03_Entrenamiento" / "train.parquet"

CLAVE_SERIE = ["store_id", "item_id"]   # una serie = una tienda + un producto
COL_FECHA = "date"
COL_VENTAS = "ventas"

# Serie elegida para aprender (ver sección 1.1 y DEC-006)
SERIE = {"store_id": "CA_3", "item_id": "FOODS_3_586"}


def cargar_tablon(ruta):
    """Lee un parquet (trae `date` como índice) y devuelve `date` como columna."""
    return pd.read_parquet(ruta).reset_index()


train = cargar_tablon(RUTA_TRAIN)
print(train.shape)
print(train[COL_FECHA].min(), "->", train[COL_FECHA].max())

## 1. Elegir una serie y graficarla

### 1.1 ¿Por qué una sola serie?
ARIMA y SARIMA son modelos **univariados**: aprenden de la historia de **una** serie y de nada más. Si tenemos 20 series, en la práctica se ajustan 20 modelos, uno por serie.

Para aprender conviene empezar con una sola, y que sea **"fácil"**:
- **Pocos ceros.** SARIMA supone que la serie se mueve alrededor de un nivel de forma más o menos continua. Una serie con muchos días en cero (demanda *intermitente*) necesita otros métodos (por ejemplo Croston), y confundiría el aprendizaje.
- **Volumen alto.** Con ventas de 2 o 3 unidades por día, el ruido tapa cualquier patrón.
- **Poca variación relativa.** Lo medimos con el **coeficiente de variación** (CV = desvío estándar / media). Un CV bajo indica una serie más estable y predecible.

Calculamos esas tres cosas para las 20 series:

In [ ]:
def resumen_series(df, clave=CLAVE_SERIE, col_valor=COL_VENTAS):
    """Una fila por serie: cantidad de días, media, % de días en cero y coeficiente de variación."""
    g = df.groupby(clave)[col_valor]
    resumen = pd.DataFrame({
        "dias": g.size(),
        "media": g.mean(),
        "pct_ceros": g.apply(lambda s: (s == 0).mean() * 100),
        "cv": g.std() / g.mean(),   # cuanto más bajo, más estable
    })
    return resumen.round(2).sort_values(["pct_ceros", "cv"])


resumen_series(train)

**Elegimos `CA_3` / `FOODS_3_586`**: empata en el primer lugar con menos ceros (0,19 %, apenas 2 días), es la de menor CV (0,24) y la segunda de mayor volumen (73,8 unidades por día de media). Es la candidata más limpia para aprender. Las series con 30-45 % de ceros quedan para más adelante.

### 1.2 De tabla a serie de tiempo
Para `statsmodels` (la librería de ARIMA/SARIMA) una serie de tiempo es una `pd.Series` cuyo **índice son fechas** con una **frecuencia declarada**. En pandas:
- `DatetimeIndex`: un índice de fechas. Permite hacer cosas como `y["2014-07"]` (todo julio de 2014) o `y.resample("MS")` (agrupar por mes).
- **Frecuencia** (`freq`): le dice a pandas cada cuánto hay un dato. `"D"` = diaria. Con `.asfreq("D")` se fuerza un dato por día: si faltara una fecha, aparece como `NaN` en vez de quedar un salto escondido.

¿Por qué importa no tener huecos? Porque los modelos de series asumen que el dato *t-1* es **el día anterior**. Si faltara un día, "ayer" en realidad sería "anteayer" y el modelo aprendería mal la memoria de la serie.

In [ ]:
def extraer_serie(df, filtro, col_fecha=COL_FECHA, col_valor=COL_VENTAS, freq="D"):
    """Devuelve una pd.Series con índice de fechas y frecuencia fija para una serie del tablón.

    filtro: dict {columna: valor} que identifica la serie, por ejemplo SERIE.
    Falla si hay fechas repetidas o huecos, en vez de dejar un problema silencioso.
    """
    mascara = pd.Series(True, index=df.index)
    for col, valor in filtro.items():
        mascara &= df[col] == valor
    serie = df.loc[mascara].set_index(col_fecha)[col_valor].sort_index()

    if not serie.index.is_unique:
        raise ValueError("Hay fechas repetidas: la serie no está bien identificada")
    serie = serie.asfreq(freq)  # un dato por período; los huecos aparecerían como NaN
    huecos = serie.isna().sum()
    if huecos:
        raise ValueError(f"La serie tiene {huecos} períodos sin dato")
    serie.name = "/".join(filtro.values())
    return serie


y = extraer_serie(train, SERIE)
print(y.index.freq, "|", len(y), "días |", y.index.min().date(), "->", y.index.max().date())
y.describe().round(1)

### 1.3 La serie completa
Lo primero que se hace con **cualquier** serie es graficarla entera. Antes de modelar, a ojo, buscamos tres cosas:
- **Tendencia:** ¿el nivel sube o baja a lo largo de los años?
- **Estacionalidad:** ¿hay un patrón que se repite cada cierto período fijo (semana, año)?
- **Cosas raras:** picos, caídas a cero, cambios bruscos de nivel.

La serie diaria es muy "nerviosa". Para ver el movimiento de fondo le superponemos **medias móviles**: el promedio de los últimos *n* días, recalculado cada día. La de 7 días borra el patrón semanal (cada ventana tiene exactamente un lunes, un martes...). La de 28 días suaviza todavía más y deja ver la forma a lo largo del año.

In [ ]:
def graficar_serie(y, ventanas=(7, 28), titulo=None):
    """Serie diaria con sus medias móviles superpuestas."""
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(y.index, y, lw=0.6, alpha=0.5, color="grey", label="diaria")
    for n in ventanas:
        # rolling(n).mean(): promedio de los últimos n días (incluido el actual)
        ax.plot(y.index, y.rolling(n).mean(), lw=1.6, label=f"media móvil {n} días")
    ax.set_title(titulo or f"Ventas diarias — {y.name}")
    ax.set_ylabel("unidades")
    ax.legend(loc="upper left")
    plt.show()


graficar_serie(y)

### 1.4 Zoom: las últimas 8 semanas
Con casi 3 años en pantalla no se ve el detalle de cada semana. Hacemos zoom sobre las últimas 8 semanas y marcamos los fines de semana para ver si el patrón semanal es visible a ojo.

In [ ]:
def graficar_zoom(y, dias=56):
    """Últimos `dias` de la serie, con sábados y domingos sombreados."""
    tramo = y.iloc[-dias:]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(tramo.index, tramo, marker="o", ms=3)
    for fecha in tramo.index[tramo.index.day_of_week >= 5]:   # 5 = sábado, 6 = domingo
        ax.axvspan(fecha - pd.Timedelta(hours=12), fecha + pd.Timedelta(hours=12), color="orange", alpha=0.15)
    ax.set_title(f"Últimos {dias} días — fines de semana sombreados")
    ax.set_ylabel("unidades")
    plt.show()


graficar_zoom(y)

### 1.5 Confirmar con números lo que se ve
Los gráficos sugieren; los números confirman. Promediamos las ventas por día de la semana (patrón semanal), por mes (patrón anual) y por año (tendencia).

*Nota:* acá usamos `day_of_week` de pandas (0 = lunes ... 6 = domingo), no la columna `wday` del tablón, que sigue la convención M5 (1 = sábado).

In [ ]:
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]

por_dia = y.groupby(y.index.day_of_week).mean().rename(index=dict(enumerate(DIAS)))
por_mes = y.groupby(y.index.month).mean()
por_anio = y.groupby(y.index.year).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
por_dia.plot.bar(ax=axes[0], title="Media por día de la semana", rot=0)
por_mes.plot.bar(ax=axes[1], title="Media por mes (todos los años juntos)", rot=0)
for ax in axes:
    ax.set_ylabel("unidades")
plt.show()

print("Media por año:", por_anio.round(1).to_dict())
print("Días con 0 ventas:", [d.date() for d in y.index[y == 0]])
print("5 días con más ventas:")
print(y.nlargest(5))

**Observación:**
- La serie está **completa**: 1.044 días seguidos (2013-01-01 a 2015-11-10), sin fechas repetidas ni huecos.
- **Patrón semanal fuerte:** el domingo (89,5 de media) y el sábado (84,7) venden bastante más que el miércoles, el día más flojo (63,2). Es una diferencia de alrededor del 40 %. En el zoom se ve: los picos caen en los fines de semana. Esto es **estacionalidad con período 7**, el `s = 7` que va a usar SARIMA.
- **Patrón anual:** la media móvil de 28 días sube en verano y cae a principios de cada año. Juntando los tres años, agosto es el mes más fuerte (87,4 de media) y enero el más flojo (62,3); mes a mes, el máximo es agosto de 2014 (90,9) y el mínimo enero de 2015 (58,5). Varios de los días récord son de verano (142 el 2015-08-15, 137 el 2014-07-26). Esto es **estacionalidad anual (período 365)**, que SARIMA no maneja bien con datos diarios; lo tratamos en el paso 2.
- **Sin tendencia clara:** la media por año casi no cambia (72,8 en 2013, 75,3 en 2014 y 73,2 en 2015, este último sin noviembre y diciembre).
- **Solo 2 ceros**, los dos un 25 de diciembre (2013 y 2014), cuando las tiendas cierran. No son demanda cero, son tienda cerrada. Ya está en los pendientes de `decisions.md`.

**Lo que nos llevamos al paso 2:** la serie tiene nivel estable, un ciclo semanal muy marcado y un ciclo anual suave. En el paso 2 vamos a **separar** esos componentes con una descomposición, para verlos por separado y medir cuánto pesa cada uno.

## 2. Descomponer la serie: tendencia, estacionalidad y ruido

### 2.1 La idea
**Descomponer** una serie es separarla en piezas más simples que, sumadas, la reconstruyen:

| Componente | Qué es | En esta serie |
|---|---|---|
| **Tendencia** (*trend*) | El nivel de fondo, que cambia despacio | Casi plana (sección 1) |
| **Estacionalidad** (*seasonal*) | Un patrón que se repite cada período fijo | Semanal (7 días) y anual (365) |
| **Residuo** (*resid*) | Lo que sobra: ruido, eventos, sorpresas | Picos sueltos, 25 de diciembre |

¿Para qué sirve? Para **medir** cuánto pesa cada pieza y decidir qué tiene que modelar SARIMA. Un SARIMA bien elegido captura la tendencia y la estacionalidad; si después de modelar el residuo todavía muestra un patrón, el modelo se quedó corto.

Las piezas se pueden combinar de dos maneras:
- **Aditiva:** `y = tendencia + estacionalidad + residuo`. El efecto del domingo es "+X unidades", siempre las mismas.
- **Multiplicativa:** `y = tendencia × estacionalidad × residuo`. El efecto del domingo es "+X %": si el nivel sube, el pico del domingo crece con él.

¿Cómo saber cuál usar? Mirando si **la amplitud del ciclo semanal crece cuando sube el nivel**. Para cada mes calculamos el nivel (la media) y la amplitud (el día de la semana más alto menos el más bajo).

In [ ]:
def amplitud_vs_nivel(y, periodo_agrupar="M"):
    """Por cada mes: nivel medio y amplitud del ciclo semanal (máximo - mínimo de las medias por día)."""
    tabla = (
        y.groupby([y.index.to_period(periodo_agrupar), y.index.day_of_week])
        .mean()
        .unstack()   # filas = meses, columnas = días de la semana
    )
    return pd.DataFrame({"nivel": tabla.mean(axis=1), "amplitud": tabla.max(axis=1) - tabla.min(axis=1)})


amp = amplitud_vs_nivel(y)
print("Correlación nivel-amplitud:", round(amp["nivel"].corr(amp["amplitud"]), 2))

ax = amp.plot.scatter(x="nivel", y="amplitud", figsize=(6, 4), title="¿El ciclo semanal crece con el nivel?")
ax.set_xlabel("nivel del mes (unidades/día)")
ax.set_ylabel("amplitud semanal (unidades)")
plt.show()

**Observación:** la correlación es casi nula (0,11). Enero de 2013 (nivel 62,1) tiene una amplitud de 30,1, y agosto de 2014 (nivel 90,1) tiene 30,2: el nivel sube casi 30 unidades y el ciclo semanal mide lo mismo. **Alcanza con el modelo aditivo** (DEC-008).

### 2.2 Descomposición clásica (`seasonal_decompose`)
Es el método más viejo y el más fácil de entender:
1. **Tendencia** = media móvil centrada de 7 días (los 3 días anteriores, el día y los 3 siguientes). Como una semana completa tiene un día de cada tipo, el patrón semanal se cancela.
2. **Estacionalidad** = para cada día de la semana, el promedio de cuánto se aparta la serie de la tendencia. Sale un único valor por día, **igual para todos los años**.
3. **Residuo** = serie − tendencia − estacionalidad.

Limitación: la media móvil centrada no se puede calcular en los 3 primeros ni en los 3 últimos días (les faltan vecinos), así que ahí la tendencia queda en `NaN`.

In [ ]:
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose, STL, MSTL

clasica = seasonal_decompose(y, model="additive", period=7)
clasica.plot().set_size_inches(14, 8)
plt.show()

# Un valor fijo por día de la semana: cuántas unidades suma o resta ese día respecto a la tendencia
efecto_dia = clasica.seasonal.groupby(clasica.seasonal.index.day_of_week).first()
print("Efecto de cada día (unidades):", efecto_dia.rename(index=dict(enumerate(DIAS))).round(1).to_dict())
print("Días sin tendencia (NaN en los bordes):", clasica.trend.isna().sum())

### 2.3 STL: una descomposición más robusta
**STL** (*Seasonal-Trend decomposition using Loess*) hace lo mismo, pero mejor en dos cosas:
- **Robusta:** con `robust=True` le da menos peso a los días raros (como el 25 de diciembre), que en la clásica contaminan la estacionalidad.
- **Flexible:** deja que el patrón semanal cambie de a poco con el tiempo, en vez de forzarlo igual en los 3 años.

Para resumir cuánto pesa cada componente usamos la **fuerza** (Hyndman y Athanasopoulos, *Forecasting: Principles and Practice*, cap. 4):

`fuerza = 1 − var(residuo) / var(componente + residuo)`

Va de 0 a 1. Cerca de 0, el componente se pierde en el ruido; cerca de 1, domina. Como referencia práctica, por encima de ~0,6 se considera fuerte.

In [ ]:
def fuerza(componente, residuo):
    """Fuerza de un componente (0 = se pierde en el ruido, 1 = domina)."""
    return max(0.0, 1 - residuo.var() / (componente + residuo).var())


stl = STL(y, period=7, robust=True).fit()
stl.plot().set_size_inches(14, 8)
plt.show()

print("Fuerza estacional semanal:", round(fuerza(stl.seasonal, stl.resid), 2))
print("Fuerza de la tendencia:   ", round(fuerza(stl.trend, stl.resid), 2))

Con un solo período (7), la "tendencia" de STL no es una línea recta: sube en verano y baja en invierno. **Está absorbiendo la estacionalidad anual**, porque nadie le dijo que existía.

### 2.4 ¿Y la estacionalidad anual?
`MSTL` (*Multiple STL*) acepta varios períodos a la vez. Le pedimos 7 y 365, y miramos con cuidado lo que devuelve.

In [ ]:
mstl = MSTL(y, periods=(7, 365)).fit()
anual = mstl.seasonal["seasonal_365"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(anual.index, anual, lw=0.8)
ax.set_title("Componente anual según MSTL")
ax.set_ylabel("unidades")
plt.show()

print("Años de datos:", round(len(y) / 365, 2))
print("Salto medio de un día al siguiente:", round(anual.diff().abs().mean(), 1), "unidades")
print("Días con mayor efecto anual:")
print(anual.abs().nlargest(5).round(1))

**Esto NO es una estacionalidad anual creíble.** Una curva anual real (más ventas en verano) debería cambiar despacio. Esta salta varias unidades de un día al siguiente, y sus valores más extremos son días sueltos: los dos 25 de diciembre (cerca de −59) y el 4 de julio de 2013 (+52).

¿Por qué pasa? Con 2,86 años de datos, cada día del calendario aparece solo 2 o 3 veces. Al estimar "el efecto del 25 de diciembre" con dos observaciones, el método copia lo que pasó esos dos días. Esto se llama **sobreajuste** (*overfitting*): el modelo memoriza el ruido en vez de aprender el patrón.

La alternativa es describir la forma anual con unas pocas curvas suaves, los **términos de Fourier**: pares seno/coseno con período de un año. Con `K` pares, la curva solo puede tener forma suave (con `K = 2` puede tener hasta dos subidas y dos bajadas por año), y se estiman `2K` números en vez de 365. La comparamos contra la media móvil de 7 días, que ya no tiene el ciclo semanal.

In [ ]:
def terminos_fourier(indice, periodo=365.25, K=2):
    """Pares seno/coseno del día del año. Sirven como variables externas para SARIMAX (paso 5)."""
    t = indice.day_of_year.to_numpy()
    columnas = {}
    for k in range(1, K + 1):
        columnas[f"sin_{k}"] = np.sin(2 * np.pi * k * t / periodo)
        columnas[f"cos_{k}"] = np.cos(2 * np.pi * k * t / periodo)
    return pd.DataFrame(columnas, index=indice)


suave = y.rolling(7, center=True).mean().dropna()   # serie sin ciclo semanal
X = terminos_fourier(suave.index, K=2)
X.insert(0, "const", 1.0)
# Regresión lineal común: busca la combinación de senos/cosenos que mejor sigue a la serie suavizada
coef, *_ = np.linalg.lstsq(X.to_numpy(), suave.to_numpy(), rcond=None)
curva_anual = pd.Series(X.to_numpy() @ coef, index=suave.index)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(suave.index, suave, lw=0.8, color="grey", label="media móvil 7 días")
ax.plot(curva_anual.index, curva_anual, lw=2, label="curva anual con Fourier (K=2)")
ax.set_ylabel("unidades")
ax.legend(loc="upper left")
plt.show()

print("Rango de la curva anual:", round(curva_anual.max() - curva_anual.min(), 1), "unidades")
print("Variación lenta que explica (R²):", round(1 - (suave - curva_anual).var() / suave.var(), 2))

**Observación:**
- **Aditiva:** la amplitud semanal no crece con el nivel (correlación 0,11), así que la descomposición aditiva es la correcta.
- **Ciclo semanal:** según la descomposición clásica, el domingo suma unas 15,5 unidades a la tendencia, el sábado 10,9, y el miércoles resta 10,5. Con STL, la fuerza estacional semanal es 0,45: moderada, porque el residuo todavía tiene mucho ruido diario.
- **Tendencia:** con STL de un solo período, la "tendencia" (fuerza 0,41) incluye la ola anual.
- **Ciclo anual:** existe (verano alto, enero bajo), pero **no se puede estimar día por día** con menos de 3 años de datos. MSTL sobreajusta: su componente anual salta en promedio 9,8 unidades de un día al siguiente y copia los 25 de diciembre y el 4 de julio de 2013. Con 2 pares de Fourier sale una curva suave de 18,5 unidades de rango, que explica el 42 % de la variación lenta (la media móvil de 7 días).

**Lo que nos llevamos al paso 3:**
- SARIMA va a modelar el ciclo **semanal** con `s = 7`.
- El ciclo **anual** entra como variables externas (términos de Fourier) en un `SARIMAX`, en el paso 5 (DEC-009). Un SARIMA con `s = 365` sobre datos diarios es inviable: tendría que estimar relaciones a 365 días de distancia con apenas 2,86 ciclos, y además es muy lento.
- En el paso 3 vamos a ver si la serie es **estacionaria**, una condición que ARIMA necesita.

## 3. Estacionariedad — parte A: diagnóstico

El paso 3 está dividido en dos partes. En esta (A) diagnosticamos si la serie original es estacionaria. En la parte B la vamos a diferenciar y a decidir cuántas diferencias necesita el modelo.

### 3.1 La idea
ARIMA aprende reglas del tipo "las ventas de hoy se parecen en tal proporción a las de ayer". Para que una regla aprendida con datos de 2013 sirva para predecir 2015, esa regla tiene que valer en todo el período. A esa propiedad se la llama **estacionariedad**.

Una serie es estacionaria cuando, a grandes rasgos, tres cosas no cambian con el tiempo:
1. **La media:** no hay tendencia; la serie oscila alrededor de un mismo nivel.
2. **La variabilidad (varianza):** las oscilaciones tienen más o menos el mismo tamaño en cualquier época.
3. **La relación entre un día y los anteriores (autocorrelación):** depende de la distancia entre los días, no de la fecha.

Dos ejemplos de series **no** estacionarias: una con tendencia (la media sube año a año) y una con estacionalidad (la media del domingo es distinta de la del miércoles: depende de qué día mires).

### 3.2 Mirarla primero
Antes de cualquier test, graficamos la **media y el desvío móviles** con una ventana larga, de 91 días (unas 13 semanas, un trimestre). La ventana larga borra el ciclo semanal y deja ver si el nivel o la variabilidad cambian a lo largo del tiempo.

In [ ]:
def graficar_media_desvio_movil(y, ventana=91):
    """Serie con su media y desvío móviles. Si la serie fuera estacionaria, las dos líneas serían casi planas."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(y.index, y, lw=0.5, color="grey", alpha=0.6, label="diaria")
    axes[0].plot(y.index, y.rolling(ventana).mean(), lw=2, label=f"media móvil {ventana} días")
    axes[0].set_ylabel("unidades")
    axes[0].legend(loc="upper left")
    axes[1].plot(y.index, y.rolling(ventana).std(), lw=2, color="tab:red", label=f"desvío móvil {ventana} días")
    axes[1].set_ylabel("unidades")
    axes[1].legend(loc="upper left")
    fig.suptitle("¿Cambian la media y la variabilidad con el tiempo?")
    plt.show()


graficar_media_desvio_movil(y)

# Lo mismo en números: media y desvío por trimestre
y.resample("QS").agg(["mean", "std"]).round(1)

### 3.3 Los tests: ADF y KPSS
A ojo se ve que la media se mueve, pero ¿alcanza para decir que la serie no es estacionaria? Para eso hay tests estadísticos. Se usan dos, porque plantean la pregunta al revés y se complementan:

| Test | Hipótesis nula (H0) | p < 0,05 significa |
|---|---|---|
| **ADF** (*Augmented Dickey-Fuller*) | La serie **no** es estacionaria: tiene una "raíz unitaria" | Rechazo H0 → **estacionaria** |
| **KPSS** (Kwiatkowski-Phillips-Schmidt-Shin) | La serie **sí** es estacionaria alrededor de un nivel | Rechazo H0 → **no estacionaria** |

**¿Qué es una raíz unitaria?** Es el caso de una serie que no tiene un nivel al que volver: cada día es el anterior más un cambio al azar, como un paseo aleatorio (el precio de una acción es el ejemplo clásico). Si un día sube, se queda arriba. Una serie estacionaria, en cambio, después de un día raro vuelve a su nivel. ADF pregunta: "¿esta serie es un paseo aleatorio?".

**¿Por qué dos tests?** Un test de hipótesis solo puede rechazar su H0; no rechazarla no prueba que sea cierta, puede ser que falten datos. Con hipótesis opuestas, si los dos apuntan al mismo lado la conclusión es firme. Si se contradicen, la serie está en una zona gris y hay que mirar más.

Una limitación: **ninguno de los dos está diseñado para detectar estacionalidad**. Miden sobre todo si hay tendencia o deriva. La estacionalidad se juzga con los gráficos y, en el paso 4, con la autocorrelación en los retardos 7, 14 y 21.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss


def test_estacionariedad(y, nombre=None, alfa=0.05):
    """Corre ADF y KPSS sobre una serie y devuelve una fila con los resultados y su lectura.

    ADF:  H0 = la serie NO es estacionaria (tiene raíz unitaria). p < alfa -> estacionaria.
    KPSS: H0 = la serie SÍ es estacionaria alrededor de un nivel. p < alfa -> no estacionaria.
    Se va a reutilizar en la parte B sobre las series diferenciadas.
    """
    y = y.dropna()   # al diferenciar aparecen NaN al principio
    adf_stat, adf_p, adf_lags, *_ = adfuller(y, autolag="AIC")   # elige cuántos retardos usar con AIC
    kpss_stat, kpss_p, kpss_lags, _ = kpss(y, regression="c", nlags="auto")   # "c" = alrededor de un nivel fijo
    return pd.DataFrame({
        "adf_estadistico": [adf_stat],
        "adf_p": [adf_p],
        "adf_retardos": [adf_lags],
        "kpss_estadistico": [kpss_stat],
        "kpss_p": [kpss_p],
        "adf_dice": ["estacionaria" if adf_p < alfa else "no estacionaria"],
        "kpss_dice": ["no estacionaria" if kpss_p < alfa else "estacionaria"],
    }, index=[nombre or y.name]).round(4)


test_estacionariedad(y, nombre="original")

**Observación:**
- **A ojo:** el nivel no deriva a lo largo de los tres años (72,4 de media en la primera mitad, 75,2 en la segunda), pero sube y baja con la estación del año: la media por trimestre va de 60,3 (primer trimestre de 2015) a 83,5 (tercer trimestre de 2014). El desvío trimestral se mueve entre 13,2 y 19,3, y es más alto en los trimestres de nivel alto.
- **ADF:** estadístico −3,57 contra un valor crítico de −2,86 al 5 %, p = 0,006. Se rechaza la raíz unitaria: la serie no es un paseo aleatorio, siempre vuelve a su nivel.
- **KPSS:** estadístico 0,419 contra 0,463 al 5 % y 0,347 al 10 %, p ≈ 0,07. No se rechaza la estacionariedad al 5 %, pero sí al 10 %. Es un caso límite.
- **Lectura conjunta:** los dos tests coinciden en lo importante, que **no hay tendencia ni deriva**, así que no hace falta la diferencia simple (`d = 0` como punto de partida). El KPSS en el límite está detectando que la media no es del todo constante, y sabemos por qué: el ciclo semanal y la ola anual, que estos tests no están diseñados para medir.
- **Detalle:** ADF eligió 21 retardos, justo 3 semanas. Para dejar residuos limpios tuvo que "comerse" el ciclo semanal con retardos. Es otra señal de que el ciclo semanal es la estructura dominante.

**Lo que nos llevamos a la parte B:** la no estacionariedad de esta serie es **estacional**, no de tendencia. En la parte B vamos a aplicar la diferencia estacional (`y[t] − y[t−7]`), y también la simple para comparar, y repetir estos tests y gráficos para decidir `d` y `D`. La ola anual no se diferencia: queda para los términos de Fourier (DEC-009).